# Downloading and processing EDDEv2 for the DL downscaling task

In [ ]:
import xarray as xr
import numpy as np

# Data source location

`s3://epa-edde-v2/`

## Download invariant files

In [ ]:
!aws s3 cp --no-sign-request s3://epa-edde-v2/EDDE_V2/static/WRF-MPI/orog.hist.MPI.EDDE-WRF.fixed.NA12.raw.nc EDDEv2/eddev2_full_orography.nc

In [ ]:
!aws s3 cp --no-sign-request s3://epa-edde-v2/EDDE_V2/static/WRF-MPI/lmask.hist.MPI.EDDE-WRF.fixed.NA12.raw.nc EDDEv2/eddev2_full_lsm.nc

## Examine the full data

In [ ]:
with xr.open_dataset('EDDEv2/eddev2_full_orography.nc') as ds:
    ds = (
        ds
        .assign_coords(
            latitude=(("y", "x"), ds.lat.values),
            longitude=(("y", "x"), ds.lon.values)
        )
        .drop_vars(["lat", "lon", "x", "y"])
        .rename_vars({"HGT": "orog"})
    )
    lon_360 = (ds.longitude + 360) % 360
    ds = ds.assign_coords(longitude=lon_360)
    ds = ds.load()
ds

In [ ]:
# Now the file is closed → safe to overwrite
ds.to_netcdf('EDDEv2/eddev2_full_orography.nc')

In [ ]:
orog = xr.open_dataset('EDDEv2/eddev2_full_orography.nc')
orog.orog.plot.contourf(x='longitude', y='latitude')

In [ ]:
with xr.open_dataset('EDDEv2/eddev2_full_lsm.nc') as ds:
    ds = (
        ds
        .assign_coords(
            latitude=(("y", "x"), ds.lat.values),
            longitude=(("y", "x"), ds.lon.values)
        )
        .drop_vars(["lat", "lon", "x", "y"])
        .rename_vars({"LANDMASK": "lsm"})
    )
    lon_360 = (ds.longitude + 360) % 360
    ds = ds.assign_coords(longitude=lon_360)
    ds = ds.load()
ds

In [ ]:
# Now the file is closed → safe to overwrite
ds.to_netcdf('EDDEv2/eddev2_full_lsm.nc')

In [ ]:
lsm = xr.open_dataset('EDDEv2/eddev2_full_lsm.nc')
lsm.lsm.plot.contourf(x='longitude', y='latitude')

## Crop EDDEv2 to NYS boundary

In [ ]:
orog = xr.open_dataset('EDDEv2/eddev2_full_orography.nc').orog

lat = orog.latitude.values
lon = orog.longitude.values

lat_min, lat_max = 38, 48
lon_min, lon_max = 278, 292

mask = (
    (lat >= lat_min) & (lat <= lat_max) &
    (lon >= lon_min) & (lon <= lon_max)
)

# Get bounding box indices
ys, xs = np.where(mask)

y0, y1 = ys.min(), ys.max() + 1
x0, x1 = xs.min(), xs.max() + 1

print(f"y: {y0}:{y1}, x: {x0}:{x1}")

In [ ]:
orog_nys_cropped = orog.isel(y=slice(y0, y1), x=slice(x0, x1))
print(orog_nys_cropped.sizes)
orog_nys_cropped.plot(x='longitude', y='latitude')

In [ ]:
orog_nys_cropped = orog.isel(y=slice(y0, y1), x=slice(x0, x1))
orog_nys_cropped.to_netcdf('EDDEv2/eddev2_nys_cropped_orography.nc')

In [ ]:
lsm = xr.open_dataset('EDDEv2/eddev2_full_lsm.nc').lsm
lsm_nys_cropped = lsm.isel(y=slice(y0, y1), x=slice(x0, x1))
lsm_nys_cropped.plot(x='longitude', y='latitude')

In [ ]:
lsm_nys_cropped.to_netcdf('EDDEv2/eddev2_nys_cropped_lsm.nc')

## Creating xesmf regridders

In [ ]:
eddev2_nys_cropped = xr.open_dataset('EDDEv2/eddev2_nys_cropped_orography.nc').copy()
urma_nys = xr.open_dataset("urma_nys_orography.nc")

### Full HR grid (256 x 288)

In [ ]:
import xesmf as xe
# We don't have to provide grid dicts, rather source files having the latitude/lat and longitude/lon coordinates can work.
regridder_eddev2_to_urma = xe.Regridder(
    eddev2_nys_cropped,
    urma_nys,
    method='bilinear',
    filename='EDDEv2/xesmf_weights_eddev2_to_urma_full_HR.nc',
    reuse_weights=False
)

In [ ]:
regridded = regridder_eddev2_to_urma(eddev2_nys_cropped.orog)
print(regridded.sizes)
regridded.plot(x='longitude', y='latitude')

### Full LR grid (256/5 x 288/5)

In [ ]:
sy = 5
sx = 5
target_grid = urma_nys.isel(
    y=slice(0,None,sy),
    x=slice(0,None,sx)
).copy()   # forces contiguous array
regridder_eddev2_to_urma = xe.Regridder(
    eddev2_nys_cropped,
    target_grid,
    method='bilinear',
    filename=f'EDDEv2/xesmf_weights_eddev2_to_urma_full_LR_{sy}x{sx}.nc',
    reuse_weights=False
)

In [ ]:
regridded = regridder_eddev2_to_urma(eddev2_nys_cropped.orog)
print(regridded.sizes)
regridded.plot(x='longitude', y='latitude')

# Examining the s3 bucket for variables

In [ ]:
!aws s3 ls --no-sign-request s3://epa-edde-v2/EDDE_V2/hourly/WRF-MPI/Historical/1985-2014/2014/ --recursive | grep '\.nc$'

In [ ]:
!aws s3 ls --no-sign-request s3://epa-edde-v2/EDDE_V2/hourly/WRF-MPI/SSP2-4.5/2025-2100/2025/ --recursive | grep '\.nc$'


In [ ]:
import xarray as xr
source_dir = '/network/rit/lab/basulab/RAW_DATA/EDDEv2_first_month_files'
vars = ["clt", "hfls", "hfss", "hur", "hus", "pr", "ps", "rlds", "rlut", "rsds", "td", "ts", "ua", "ustar", "va", "wdirs", "wspds"]
for var in vars:
    ds = xr.open_dataset(f"{source_dir}/{var}.SSP2-4.5.mpi.EDDE-WRF.1hr.NA12.2025-01.raw.nc")
    candidates = [v for v in ds.data_vars if v not in {"lat", "lon", "time", "mtime"}]
    data_var = candidates[0] if candidates else next(iter(ds.data_vars))
    print(var, data_var, ds[data_var].attrs.get("long_name"))
    ds.close()